# 4.7 — Compare All Arms

> Run `4.0` then any of `4.1`-`4.6` first. This notebook only **reads** their JSON files.

Collects whichever arms have run, builds the comparison table and Figure 1, and saves the
winning head for Sections 7 (Reptile) and 8 (KD).

## How to read the table

1. **Compare paid against free.** SmartReplay (4.5) vs WeightAlign (4.2) and LwF (4.4).
   If the paid method only matches the free ones, the memory is not earning its cost.
2. **Everything against the ceiling.** "71% vs an 85% ceiling" is a sentence; "71%" is not.
3. **Naive must collapse.** If it doesn't, the protocol is broken and nothing else means
   anything.

## Setup

In [ ]:
# ── ULAL bootstrap — self-contained, no Drive dependency ────────────────────
# One shallow git clone instead of hundreds of API calls. subprocess takes a
# LIST, never a shell string, so an '&' in a branch name is safe.
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden): ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

# ── Verify the clone contains what this notebook needs ──────────────────────
# The clone reflects what is ON GITHUB. Local commits you never pushed are
# invisible here, and the symptom is a confusing ModuleNotFoundError later.
_need = ["src/__init__.py", "src/bootstrap.py", "src/imports.py",
         "src/data/cache.py", "src/cl/trainer.py",
         "src/models/temporal_head.py", "src/utils/probe.py"]
_miss = [f for f in _need if not os.path.exists(os.path.join(ROOT, f))]

if _miss:
    print("ERROR — the cloned repo is missing:")
    for f in _miss:
        print("   ", f)
    print("")
    print("These exist locally but are not on the '" + BRANCH + "' branch on GitHub.")
    print("On your machine run:")
    print("    git add -A")
    print("    git commit -m 'sync src'")
    print("    git push origin " + BRANCH)
    print("then re-run this cell.")
    raise RuntimeError("src/ incomplete on GitHub — push your commits first.")

print("src/ complete (" + str(len(_need)) + " key files present)")

In [ ]:
# ── Drive + dataset + paths ─────────────────────────────────────────────────
# Exports ULAL_* env vars that config.py reads. Nothing is written to disk and
# no kernel restart is needed.
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
# ── Global label space (src/data/cache.py) ──────────────────────────────────
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]     # Sections 4-5 stream over these
FEWSHOT_TASK = 3          # Section 7 probe only

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

In [ ]:
# ── Load cached frame features (src/data/cache.py) ──────────────────────────
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

## Collect whatever has run

In [ ]:
ARM_FILES = [
    ("Naive",       "stage1_arm_naive.json"),
    ("WeightAlign", "stage1_arm_weight_align.json"),
    ("EWC",         "stage1_arm_ewc.json"),
    ("LwF",         "stage1_arm_lwf.json"),
    ("SmartReplay", "stage1_arm_smart_replay.json"),
    ("Replay+LwF",  "stage1_arm_replay_lwf.json"),
]

ARMS, missing = [], []
for name, fn in ARM_FILES:
    p = f"{cfg.RESULTS_DIR}/{fn}"
    if os.path.exists(p):
        ARMS.append((name, json.load(open(p))))
    else:
        missing.append(name)

CEILING = ARMS[0][1].get("ceiling") if ARMS else None
print(f"Loaded {len(ARMS)} arm(s): {', '.join(n for n, _ in ARMS)}")
if missing:
    print(f"Not yet run: {', '.join(missing)}")
if not ARMS:
    raise RuntimeError("No arm results found — run 4.1-4.6 first.")

## Per-arm reports

In [ ]:
for name, d in ARMS:
    cl_report(np.array(d["matrix"]), name, d["task_names"][:len(d["matrix"])])

## Summary table

In [ ]:
hdr = f"{'Method':<14}{'Base':>9}{'Final AA':>10}{'BWT':>10}{'Forget':>9}"
hdr += f"{'vs ceiling':>12}" if CEILING else ""
hdr += f"{'Memory/cls':>14}"
print(hdr); print("-" * len(hdr))

for name, d in sorted(ARMS, key=lambda x: x[1]["metrics"]["AA"]):
    m   = d["metrics"]
    bpc = d["bytes_per_class"]
    mem = "0" if bpc == 0 else f"{bpc/1e3:.0f} KB"
    line = f"{name:<14}{m['base_acc_final']:>8.2%}{m['AA']:>10.2%}{m['BWT']:>+10.2%}{m['forgetting']:>+9.2%}"
    line += f"{m['AA'] - CEILING:>+11.2%} " if CEILING else ""
    print(line + f"{mem:>13}")

if CEILING:
    print("-" * len(hdr))
    print(f"{'Joint ceiling':<14}{'':>9}{CEILING:>10.2%}{'':>10}{'':>9}{'':>12}{'all data':>14}")

print("\nFree methods : WeightAlign, LwF        (0 bytes)")
print("Paid methods : SmartReplay, Replay+LwF")
print("-> If a paid method only matches a free one, the memory is not earning its cost.")

## Figure 1

In [ ]:
palette = {"Naive": "#DC2626", "WeightAlign": "#D97706", "EWC": "#7C3AED",
           "LwF": "#0891B2", "SmartReplay": "#2563EB", "Replay+LwF": "#16A34A"}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# (a) base-class retention
for name, d in ARMS:
    R = np.array(d["matrix"])
    axes[0].plot(range(len(R)), R[:, 0], marker="o", lw=2, label=name,
                 color=palette.get(name))
if CEILING:
    axes[0].axhline(CEILING, ls="--", c="#166534", lw=1.5, label="ceiling")
axes[0].set_xticks(range(len(ARMS[0][1]["matrix"])))
axes[0].set_xticklabels([f"after {m}" for m in ARMS[0][1]["task_names"][:len(ARMS[0][1]["matrix"])]])
axes[0].set_ylabel("Test accuracy on BASE classes")
axes[0].set_title("(a) Base retention as tasks arrive", fontsize=11)
axes[0].set_ylim(0, 1.05); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# (b) AA vs BWT
nm = [n for n, _ in ARMS]
aa = [d["metrics"]["AA"]  for _, d in ARMS]
bw = [d["metrics"]["BWT"] for _, d in ARMS]
w, xs = 0.35, range(len(ARMS))
axes[1].bar([i - w/2 for i in xs], aa, w, label="Average Accuracy", color="#16A34A")
axes[1].bar([i + w/2 for i in xs], bw, w, label="BWT (neg = forgetting)", color="#DC2626")
if CEILING:
    axes[1].axhline(CEILING, ls="--", c="#166534", lw=1.5, label="ceiling")
axes[1].axhline(0, c="gray", lw=0.8)
axes[1].set_xticks(list(xs)); axes[1].set_xticklabels(nm, rotation=20, ha="right", fontsize=8)
axes[1].set_title("(b) Final metrics", fontsize=11)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, axis="y")

# (c) accuracy vs memory — the preview of Section 5
for name, d in ARMS:
    x = max(d["bytes_per_class"] / 1e3, 0.5)      # 0.5 KB stands in for "free" on log axis
    axes[2].scatter(x, d["metrics"]["AA"], s=110, color=palette.get(name),
                    label=name, zorder=3, edgecolor="white", linewidth=1.5)
    axes[2].annotate(name, (x, d["metrics"]["AA"]), textcoords="offset points",
                     xytext=(7, 5), fontsize=8)
if CEILING:
    axes[2].axhline(CEILING, ls="--", c="#166534", lw=1.5)
axes[2].set_xscale("log"); axes[2].set_xlabel("Memory per class (KB, log) — 0.5 = free")
axes[2].set_ylabel("Final Average Accuracy")
axes[2].set_title("(c) Accuracy vs memory\n(Section 5 fills in the curve)", fontsize=11)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/stage1_cl_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {cfg.RESULTS_DIR}/stage1_cl_comparison.png")

## Pick the winner

The winning head feeds Section 7 (Reptile) and Section 8 (KD).

In [ ]:
winner, wd = max(ARMS, key=lambda x: x[1]["metrics"]["AA"])
ckpt_map = {"Naive": "head_naive.pt", "WeightAlign": "head_weight_align.pt",
            "EWC": "head_ewc.pt", "LwF": "head_lwf.pt",
            "SmartReplay": "head_smart_replay.pt", "Replay+LwF": "head_replay_lwf.pt"}

src_ckpt = f"{cfg.CKPT_DIR}/{ckpt_map[winner]}"
if os.path.exists(src_ckpt):
    shutil.copyfile(src_ckpt, f"{cfg.CKPT_DIR}/head_cl_best.pt")
    print(f"Copied {ckpt_map[winner]} -> head_cl_best.pt")

summary = {
    "stage": "1_continual_learning",
    "winner": winner,
    "ceiling": CEILING,
    "arms": {n: {"metrics": d["metrics"], "bytes_per_class": d["bytes_per_class"],
                 "config": d["config"]} for n, d in ARMS},
}
with open(f"{cfg.RESULTS_DIR}/stage1_cl_results.json", "w") as f:
    json.dump(summary, f, indent=2)
with open(f"{cfg.RESULTS_DIR}/stage1_winner.json", "w") as f:
    json.dump({"winner": winner, **wd["metrics"],
               "bytes_per_class": wd["bytes_per_class"]}, f, indent=2)

print(f"\nWinner: {winner}")
print(f"  Final AA : {wd['metrics']['AA']:.2%}" + (f"  (ceiling {CEILING:.2%})" if CEILING else ""))
print(f"  BWT      : {wd['metrics']['BWT']:+.2%}")
print(f"  Memory   : {wd['bytes_per_class']/1e3:.0f} KB/class")
print(f"\nSaved -> {cfg.RESULTS_DIR}/stage1_cl_results.json")

---

## Done — next: Section 5

**The three questions this table answers**

1. *Does replay beat the free methods?* SmartReplay vs WeightAlign / LwF.
2. *Does free KD help on top of paid replay?* 4.6 vs 4.5 — same memory, only the
   distillation differs, so any gap is attributable to LwF alone.
3. *How far from the ceiling is the best method?* That gap is what Sections 5-9 chip at.

**Next:** `5_memory_ablation/5.0_memory_budget.ipynb` — sweep 1/5/15/50 exemplars x
{random, hard, diverse} and fill in panel (c) properly. That curve is the project's main
result.